In [ ]:
# Cell 1: Clone repo and install deps
!git clone https://github.com/TencentARC/GFPGAN.git
%cd GFPGAN

!pip install basicsr facexlib realesrgan -q
!pip install -r requirements.txt -q
!python setup.py develop -q

Cloning into 'GFPGAN'...
remote: Enumerating objects: 527, done.
remote: Total 527 (delta 0), reused 0 (delta 0), pack-reused 527 (from 1)
Receiving objects: 100% (527/527), 5.41 MiB | 24.11 MiB/s, done.
Resolving deltas: 100% (250/250), done.
/content/GFPGAN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 17.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 20.5 MB/s eta 0:00:00
/usr/local/lib/python

In [ ]:
# Cell 2: Download GFPGANv1.4 (recommended) + RealESRGAN background upscaler
import os
os.makedirs('experiments/pretrained_models', exist_ok=True)

# GFPGAN v1.4 model
!wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth \
    -P experiments/pretrained_models

# RealESRGAN background enhancer (optional but recommended)
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth \
    -P experiments/pretrained_models

In [ ]:
# Cell 1: Upload video
from google.colab import files
import os

os.makedirs('inputs/video_frames', exist_ok=True)
os.makedirs('results/restored_frames', exist_ok=True)

uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f"Uploaded: {video_filename}")

Saving 2026_06_13_05.37.42.mp4 to 2026_06_13_05.37.42.mp4
Uploaded: 2026_06_13_05.37.42.mp4


In [ ]:
# Cell 2: Extract frames (adjust fps to control quality vs speed)
# fps=24 = full quality | fps=12 = faster but choppier
INPUT_VIDEO = video_filename
FPS = 12  # <-- change this

!ffmpeg -i "{INPUT_VIDEO}" -qscale:v 1 -qmin 1 -qmax 1 \
    -vsync 0 inputs/video_frames/frame%08d.png -hide_banner -loglevel warning

frame_count = len(os.listdir('inputs/video_frames'))
print(f"✅ Extracted {frame_count} frames at {FPS} fps")

✅ Extracted 605 frames at 12 fps


In [ ]:
# Cell — Patch basicsr (run this BEFORE Cell 3)
!sed -i \
  's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' \
  /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

print("✅ Patch applied")

✅ Patch applied


In [ ]:
# Cell 3: Restore all frames (this is the slow part — ~1–3 sec/frame on T4)
!python inference_gfpgan.py \
    -i inputs/video_frames \
    -o results \
    -v 1.4 \
    -s 2 \
    --bg_upsampler realesrgan \
    --only_center_face   # remove this line if multiple faces in frame

print("✅ Restoration complete!")

Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth" to /usr/local/lib/python3.12/dist-packages/weights/RealESRGAN_x2plus.pth

100% 64.0M/64.0M [00:00<00:00, 314MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth" to /content/GFPGAN/gfpgan/weights/detection_Resnet50_Final.pth

100% 104M/104M [00:00<00:00, 288MB/s] 
Downloading: "https://github.com/xinntao/facexlib/releases/download/

In [ ]:
# Cell 4: Merge restored frames back into video
OUTPUT_VIDEO = "gfpgan_output.mp4"

# Step A: frames → silent video
!ffmpeg -r {FPS} -i results/restored_imgs/frame%08d.png \
    -c:v libx264 -crf 17 -pix_fmt yuv420p \
    temp_silent.mp4 -hide_banner -loglevel warning

# Step B: add original audio back
!ffmpeg -i temp_silent.mp4 -i "{INPUT_VIDEO}" \
    -c:v copy -c:a aac -map 0:v:0 -map 1:a:0 \
    -shortest "{OUTPUT_VIDEO}" -hide_banner -loglevel warning

print(f"✅ Output: {OUTPUT_VIDEO}")

✅ Output: gfpgan_output.mp4
